In [60]:
import pandas as pd
import numpy as np

In [61]:
file_path=r"C:/Users/DELL/Documents/EduPro Learner Intelligence Dashboard Demographics & Course Enrollment Analytics/EduPro Online Platform.xlsx"

In [62]:
import os
print(f"File exists: {os.path.exists(file_path)}")

File exists: True


In [63]:
users=pd.read_excel(file_path,sheet_name="Users")
teachers=pd.read_excel(file_path,sheet_name="Teachers")
courses=pd.read_excel(file_path,sheet_name="Courses")
transactions=pd.read_excel(file_path,sheet_name="Transactions")

print(f"{len(users)} users")
print(f"{len(teachers)} teachers")
print(f"{len(courses)} courses")
print(f"{len(transactions)} transactions")


3000 users
60 teachers
60 courses
10000 transactions


In [64]:
print(f"USERS: \n{users.head()}")
print(f"TEACHERS: \n{teachers.head()}")
print(f"COURSES: \n{courses.head()}")
print(f"TRANSACTIONS: \n{transactions.head()}")

USERS: 
   UserID        UserName  Age  Gender                             Email
0  U00001    wilsonjordan   15    Male            patricia27@hotmail.com
1  U00002        angela22   29  Female             hallrandy@hotmail.com
2  U00003  morrisonamanda   33  Female               ganderson@yahoo.com
3  U00004       fthornton   23  Female  christensencatherine@outlook.com
4  U00005  fostergeoffrey   21    Male            karenfuentes@yahoo.com
TEACHERS: 
  TeacherID         TeacherName  Age  Gender          Expertise  \
0   TC00001  Leonard Montgomery   44  Female      Cybersecurity   
1   TC00002            Jill Day   32  Female  Digital Marketing   
2   TC00003        Amber Torres   32    Male             Design   
3   TC00004        Kristi Scott   34  Female   Machine Learning   
4   TC00005      David Williams   34    Male            Finance   

   YearsOfExperience  TeacherRating  
0                  6           3.24  
1                  9           4.14  
2                  4      

In [65]:
print(f"Missing values in Users: \n{users.isnull().sum()}")
print("====================================================")
print(f"Missing values in Courses: \n{courses.isnull().sum()}")
print("====================================================")
print(f"Missing values in Transactions: \n{transactions.isnull().sum()}")


Missing values in Users: 
UserID      0
UserName    0
Age         0
Gender      0
Email       0
dtype: int64
Missing values in Courses: 
CourseID          0
CourseName        0
CourseCategory    0
CourseType        0
CourseLevel       0
CoursePrice       0
CourseDuration    0
CourseRating      0
dtype: int64
Missing values in Transactions: 
TransactionID      0
UserID             0
CourseID           0
TransactionDate    0
Amount             0
PaymentMethod      0
TeacherID          0
dtype: int64


In [66]:
original_count=len(transactions)
transactions=transactions.dropna(subset=['UserID','CourseID'])
print(f"Removed {original_count-len(transactions)} rows with missing ID")

Removed 0 rows with missing ID


In [67]:
transactions['TransactionDate']=pd.to_datetime(transactions['TransactionDate'])
print(f"Date Range: {transactions['TransactionDate'].min()} to {transactions['TransactionDate'].max()}")

Date Range: 2025-01-01 00:00:00 to 2025-12-30 00:00:00


In [68]:
merged=transactions.merge(courses,on='CourseID',how='left')
print(f"After adding courses: {merged.shape}")

merged=merged.merge(users,on='UserID',how='left')
print(f"After adding users: {merged.shape}")

merged=merged.merge(teachers,on='TeacherID',how='left')
print(f"After adding teachers: {merged.shape}")

print(f"Total Records Merged: {len(merged)}")

After adding courses: (10000, 14)
After adding users: (10000, 18)
After adding teachers: (10000, 24)
Total Records Merged: 10000


In [69]:
merged=merged.rename(columns={
    'Age_x':'Age',
    'Gender_x':'Gender'
})
merged=merged.drop(columns=['Age_y','Gender_y'])
print(merged['Age'])
print(merged['Gender'])

0       33
1       33
2       33
3       23
4       23
        ..
9995    31
9996    28
9997    27
9998    23
9999    35
Name: Age, Length: 10000, dtype: int64
0       Female
1       Female
2       Female
3       Female
4       Female
         ...  
9995    Female
9996    Female
9997      Male
9998      Male
9999    Female
Name: Gender, Length: 10000, dtype: str


In [70]:
def age_band(age):
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Under 18'
    elif age <= 25:
        return '18-25'
    elif age <= 35:
        return '26-35'
    elif age <= 45:
        return '36-45'
    else:
        return '45+'

merged['AgeBand'] = merged['Age'].apply(age_band)
print("Age Band Distribution:")
print(merged['AgeBand'].value_counts())

Age Band Distribution:
AgeBand
26-35       4799
18-25       3732
Under 18    1469
Name: count, dtype: int64


In [74]:
merged['Year']=merged['TransactionDate'].dt.year
merged['Month']=merged['TransactionDate'].dt.month
merged['Day']=merged['TransactionDate'].dt.day
merged['WeekDay']=merged['TransactionDate'].dt.day_name()
merged['YearMonth']=merged['TransactionDate'].dt.strftime("%Y-%m")

print(f"Year present: {merged['Year'].unique()}")

Year present: [2025]


In [78]:
keys_columns=['UserID', 'CourseID', 'CourseName', 'CourseCategory', 'CourseLevel', 'Gender', 'AgeBand']
for col in keys_columns:
    missing=merged[col].isnull().sum()
    if missing>0:
        print(f"{col}:{missing} missing values")
    else:
        print(f"{col}: complete columns")
duplicated=merged.duplicated(subset=['UserID','CourseID','TransactionDate']).sum()
print(f"\nDuplicated records: {duplicated}")

UserID: complete columns
CourseID: complete columns
CourseName: complete columns
CourseCategory: complete columns
CourseLevel: complete columns
Gender: complete columns
AgeBand: complete columns

Duplicated records: 0


In [79]:
output_path = r"C:/Users/DELL/Documents/EduPro Learner Intelligence Dashboard Demographics & Course Enrollment Analytics/merged_data.csv"

merged.to_csv(output_path, index=False)
print(f"Saved merged data to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

Saved merged data to: C:/Users/DELL/Documents/EduPro Learner Intelligence Dashboard Demographics & Course Enrollment Analytics/merged_data.csv
File size: 2.24 MB


In [80]:
merged.head()

,TransactionID,UserID,CourseID,TransactionDate,Amount,PaymentMethod,TeacherID,CourseName,CourseCategory,CourseType,...,TeacherName,Expertise,YearsOfExperience,TeacherRating,AgeBand,Year,Month,Day,WeekDay,YearMonth
0,TT00001,U00003,CR00016,2025-10-25,0.0,PayPal,TC00040,Digital Marketing,Marketing,Free,...,Kimberly Miller,Cybersecurity,24,4.58,26-35,2025,10,25,Saturday,2025-10
1,TT00002,U00003,CR00037,2025-01-13,0.0,PayPal,TC00040,Scrum Essentials,Project Management,Free,...,Kimberly Miller,Cybersecurity,24,4.58,26-35,2025,1,13,Monday,2025-01
2,TT00003,U00003,CR00019,2025-03-28,0.0,Bank Transfer,TC00040,Content Marketing,Marketing,Free,...,Kimberly Miller,Cybersecurity,24,4.58,26-35,2025,3,28,Friday,2025-03
3,TT00004,U00004,CR00048,2025-06-02,0.0,Bank Transfer,TC00040,AI Ethics,Artificial Intelligence,Free,...,Kimberly Miller,Cybersecurity,24,4.58,18-25,2025,6,2,Monday,2025-06
4,TT00005,U00004,CR00060,2025-08-10,0.0,PayPal,TC00042,Content Creation,Digital Marketing,Free,...,Yolanda Levine,Machine Learning,21,4.97,18-25,2025,8,10,Sunday,2025-08
